# **The Chat Format**

In this notebook, you will explore how you can utilize the chat format to have extended conversations with chatbots personalized or specialized for specific tasks or behaviors.

## Setup

In [ ]:
from openai import OpenAI
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [ ]:
client = OpenAI(                            # Creates a client bound to your key.
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

# Helper #1: single-prompt call

def get_completion(prompt, model="gpt-3.5-turbo", temperature=0): 
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature, 
    )
    return response.choices[0].message.content

# Helper #2: multi-message call

def get_completion_from_messages(message, model="gpt-3.5-turbo", temperature=0): 
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature, 
    )
    return response.choices[0].message.content

In [ ]:
# Conversation you send
messages =  [  
{'role':'system', 'content':'You are an assistant that speaks like Shakespeare.'},    
{'role':'user', 'content':'tell me a joke'},   
{'role':'assistant', 'content':'Why did the chicken cross the road'},   
{'role':'user', 'content':'I don\'t know'}  ]

In [ ]:
response = get_completion_from_messages(messages, temperature=1)
print(response)

In [ ]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},    
{'role':'user', 'content':'Hi, my name is Isa'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

In [ ]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},    
{'role':'user', 'content':'Yes,  can you remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

In [ ]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},
{'role':'user', 'content':'Hi, my name is Isa'},
{'role':'assistant', 'content': "Hi Isa! It's nice to meet you. \
Is there anything I can help you with today?"},
{'role':'user', 'content':'Yes, you can remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

# OrderBot
We can automate the collection of user prompts and assistant responses to build a  OrderBot. The OrderBot will take orders at a pizza restaurant. 

In [ ]:
def collect_messages(_):
    prompt = inp.value_input
    inp.value = ''
    context.append({'role':'user', 'content':f"{prompt}"})
    response = get_completion_from_messages(context) 
    context.append({'role':'assistant', 'content':f"{response}"})
    panels.append(
        pn.Row('User:', pn.pane.Markdown(prompt, width=600)))
    panels.append(
        pn.Row('Assistant:', pn.pane.Markdown(response, width=600, styles={'background-color': '#F6F6F6'})))
 
    return pn.Column(*panels)


In [ ]:
import panel as pn  # GUI
pn.extension()

panels = [] # collect display 

context = [ {'role':'system', 'content':"""
You are OrderBot, an automated service to collect orders for a pizza restaurant. \
You first greet the customer, then collects the order, \
and then asks if it's a pickup or delivery. \
You wait to collect the entire order, then summarize it and check for a final \
time if the customer wants to add anything else. \
If it's a delivery, you ask for an address. \
Finally you collect the payment.\
Make sure to clarify all options, extras and sizes to uniquely \
identify the item from the menu.\
You respond in a short, very conversational friendly style. \
The menu includes \
pepperoni pizza  12.95, 10.00, 7.00 \
cheese pizza   10.95, 9.25, 6.50 \
eggplant pizza   11.95, 9.75, 6.75 \
fries 4.50, 3.50 \
greek salad 7.25 \
Toppings: \
extra cheese 2.00, \
mushrooms 1.50 \
sausage 3.00 \
canadian bacon 3.50 \
AI sauce 1.50 \
peppers 1.00 \
Drinks: \
coke 3.00, 2.00, 1.00 \
sprite 3.00, 2.00, 1.00 \
bottled water 5.00 \
"""} ]  # accumulate messages


inp = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Chat!")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=300),
)

dashboard

In [ ]:
messages =  context.copy()
messages.append(
{'role':'system', 'content':'create a json summary of the previous food order. Itemize the price for each item\
 The fields should be 1) pizza, include size 2) list of toppings 3) list of drinks, include size   4) list of sides include size  5)total price '},    
)
 #The fields should be 1) pizza, price 2) list of toppings 3) list of drinks, include size include price  4) list of sides include size include price, 5)total price '},    

response = get_completion_from_messages(messages, temperature=0)
print(response)

## Try experimenting on your own!

You can modify the menu or instructions to create your own orderbot!

# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

In [ ]:
messages = context.copy()
messages.append({
  "role": "user",
  "content": """From the prior conversation, produce ONLY valid JSON summarizing the order. 
Use the MENU PRICES provided in the system prompt; never invent items, sizes, or prices. 
If a size is missing, set it to null and add a note.
Schema:
{
  "pizzas": [{"name": str, "size": str|null, "qty": int, "unit_price": number, "subtotal": number, "toppings": [str]}],
  "sides":  [{"name": str, "size": str|null, "qty": int, "unit_price": number, "subtotal": number}],
  "drinks": [{"name": str, "size": str|null, "qty": int, "unit_price": number, "subtotal": number}],
  "total": number,
  "notes": [str]
}
Rules: 
- Prices must match the menu. 
- Subtotals = qty * unit_price. 
- total = sum of all subtotals. 
- Output JSON only, no prose."""
})


In [ ]:
messages = context.copy()
messages.append({
  "role": "user",
  "content": """Create ONLY a valid JSON object for the last order, adhering to this exact schema:
{
  "pizzas": [{"name": "pepperoni|cheese|eggplant", "size": "large|medium|small|null", "qty": int, "unit_price": number, "toppings": [ "extra cheese|mushrooms|sausage|canadian bacon|AI sauce|peppers" ], "subtotal": number}],
  "sides":  [{"name": "fries|greek salad", "size": "large|medium|small|null", "qty": int, "unit_price": number, "subtotal": number}],
  "drinks": [{"name": "coke|sprite|bottled water", "size": "large|medium|small|null", "qty": int, "unit_price": number, "subtotal": number}],
  "unknown_items": [str],
  "total": number
}
Constraints:
- Use only menu items and prices from the system prompt.
- If an item not in the menu was requested, move it to "unknown_items" and exclude it from totals.
- If size was not specified, set size=null and choose the price that matches the default size you used during the conversation; if no default, set unit_price=null and subtotal=0.
- Round all money to two decimals. 
Return JSON only."""
})


Summary

ariant A (baseline): Usually produced valid JSON and correct arithmetic when the order included explicit sizes. When sizes were omitted (e.g., “one pepperoni”), the model sometimes guessed a size and price. Adding the “set size to null and add a note” instruction helped reduce that guessing. Output was succinct and easy to parse. Occasional issue: mixing prose with JSON if temperature > 0.

Variant B (strict schema): This performed best for data cleanliness. By enumerating allowed item names/sizes and adding rounding rules, it eliminated most hallucinations. Unknown items were routed into unknown_items, preventing inflated totals. The trade-off: if the conversation used a non-standard variant (e.g., “diet coke”), the item got excluded and required a follow-up UI nudge. Still, for production-style parsing, this was the most robust.

Variant C (auditor + self_check): Produced JSON close to Variant A or B, but the extra self_check made it easy to gate the result in code (e.g., if menu_prices_used=false, we can ask the user to confirm). It caught missing sizes reliably. This variant is a nice middle ground when you want observability without a heavy schema.